# Preprocessing

## Para empezar (una sola vez)

Estas 2 celdas solo se necesitan correr una sola vez, para instalar las dependencias y crear el dataset raw pero en formato parquet.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os
import pandas as pd

EXCEL_PATH = "../data/raw/dataset-dropout.xlsx"
PARQUET_PATH = "../data/raw/dataset-dropout.parquet"

if os.path.exists(PARQUET_PATH):
    print("Cargando de parquet...")
    df = pd.read_parquet(PARQUET_PATH)
else:
    print("No hay parquet, cargando de excel y creando parquet")
    # Read using the Rust-backed calamine engine for speed
    df = pd.read_excel(EXCEL_PATH, engine="calamine")
    df.to_parquet(PARQUET_PATH, index=False)
    print("Parquet creado...")

print(f"shape: {df.shape[0]:,} rows, {df.shape[1]} features")
df.head()

## Cargar datos de Parquet

In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Configuración visual de Pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

# Carga de datos crudos
DATA_PATH = Path("../data/raw/dataset-dropout.parquet")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_parquet(DATA_PATH)
df = df_raw.copy()
df.head()

,student.id,generation,educational.model,level,gender,age,max.degree.parents,father.education.complete,father.education.summary,mother.education.complete,mother.education.summary,parents.exatec,father.exatec,mother.exatec,tec.no.tec,foreign,zone.type,first.generation,school,program,region,PNA,admission.test,online.test,english.evaluation,admission.rubric,general.math.eval,retention,FTE,scholarship.perc,scholarship.type,loan.perc,total.scholarship.loan,school.cost,id.school.origin,socioeconomic.level,social.lag,average.first.period,failed.subject.first.period,dropped.subject.first.period,dropout.semester,physical.education,cultural.diffusion,student.society,total.life.activities,athletic.sports,art.culture,student.society.leadership,life.work.mentoring,wellness.activities
0,1,AD15,0,High School,Male,15,No information,No information,No information,No information,No information,No information,No information,No information,TEC,Local,No information,Does not apply,High school,PBB,DR,75.00,793,0,0,NaN,Does not apply,0,1.00,0.00,No scholarship,0.00,0.000000,High cost,School 2602,No information,No information,NaN,NaN,NaN,1,No information,No information,No information,Does not apply,Does not apply,Does not apply,Does not apply,Does not apply,Does not apply
1,2,AD14,0,Undergraduate,Male,19,No information,No information,No information,No information,No information,No information,No information,No information,TEC,Yes: National,No information,No information,EN,LDE,RO,87.00,Does not apply,0,6,NaN,75.5,1,1.08,0.00,No scholarship,0.00,0.000000,High cost,School 5332,No information,No information,NaN,NaN,NaN,0,0,0,0,Does not apply,Does not apply,Does not apply,Does not apply,Does not apply,Does not apply
2,3,AD18,0,Undergraduate,Male,23,Undergraduate degree,Received undergraduate degree,Undergraduate degree,"Attended university, but did not graduate",No degree,No,No,No,TEC,Yes: National,No information,No,EIC,IID,RM,83.76,Does not apply,0,6,NaN,86.5,0,1.08,0.35,Traditional,0.25,0.599998,High cost,School 5332,No information,No information,NaN,NaN,NaN,4,Does not apply,Does not apply,Does not apply,1,0,0,0,1,0
3,4,AD14,0,Undergraduate,Male,19,No information,No information,No information,No information,No information,No information,No information,No information,NO TEC,Yes: Foreigner,No information,No information,EN,LDE,RO,71.00,1220,0,6,24.0,20,1,1.08,0.00,No scholarship,0.00,0.000000,Not defined,School 5604,No information,No information,NaN,NaN,NaN,0,0,0,0,Does not apply,Does not apply,Does not apply,Does not apply,Does not apply,Does not apply
4,5,AD14,0,Undergraduate,Male,17,No information,No information,No information,No information,No information,No,No,No,TEC,Yes: National,No information,No information,EIC,IIS,RM,96.86,1529,0,6,NaN,98.5,1,1.08,0.90,Traditional,0.00,0.900003,High cost,School 5332,No information,No information,NaN,NaN,NaN,0,1,1,1,Does not apply,Does not apply,Does not apply,Does not apply,Does not apply,Does not apply


## Filtros Estructurales y Definición del Target
Aquí aislamos a la población de Profesional (Undergraduate) antes de generar cualquier índice categórico. Esto evita crear categorías vacías en el índice de escuelas (school_code) que causarían problemas de convergencia en el modelo multinivel. También definimos la variable objetivo y la variable binaria del modelo educativo.

In [3]:
# Aislar población de Profesional y limpiar la columna
df = df[df["level"] == "Undergraduate"].copy()
df = df.drop(columns=["level"])

# Eliminar nulos en retención y crear target de deserción
df = df.dropna(subset=["retention"])
df["dropout"] = 1 - df["retention"]

# Definir la época (era_code: 0 = Pre-Tec21, 1 = Tec21)
tec21_generations = {"AD19", "AD20"}
df["era_code"] = (df["generation"].isin(tec21_generations) | df["educational.model"].eq(1)).astype("int8")

# Generar el índice de escuelas solo con las observaciones restantes
school_categories = sorted(df["school"].dropna().unique())
df["school_code"] = pd.Categorical(df["school"], categories=school_categories, ordered=False).codes.astype("int16")

## Ingeniería de Variables y Unificación
Resolvemos la fragmentación del registro de actividades extracurriculares. En lugar de lidiar con columnas antiguas y nuevas llenas de ceros estructurales, consolidamos todo en activity_count_unified. Además, eliminamos explícitamente las variables del primer periodo (que solo existen en Tec21) para que el modelo logístico pueda comparar los mismos coeficientes en ambas épocas sin caer en multicolinealidad.

In [4]:
# Unificación de actividades (LiFE y modelo previo)
activity_cols = [
    "physical.education", "cultural.diffusion", "student.society",
    "athletic.sports", "art.culture", "student.society.leadership", 
    "life.work.mentoring", "wellness.activities"
]

# Reemplazar texto por NaN y convertir a numérico para sumar
activities_numeric = df[activity_cols].replace(["No information", "Does not apply"], np.nan)
activities_numeric = activities_numeric.apply(pd.to_numeric, errors="coerce")

df["activity_count_unified"] = activities_numeric.sum(axis=1, min_count=1)

# Limpieza de textos a numéricos en variables académicas/financieras
numeric_conversions = ["admission.test", "general.math.eval", "scholarship.perc", "loan.perc"]
for col in numeric_conversions:
    df[col] = pd.to_numeric(df[col].replace(["No information", "Does not apply"], np.nan), errors="coerce")

# Eliminar variables exclusivas de Tec21 para evitar pesos predictivos nulos en Pre-Tec21
tec21_exclusive_cols = [
    "average.first.period", "failed.subject.first.period", "dropped.subject.first.period"
]
df = df.drop(columns=tec21_exclusive_cols + activity_cols, errors="ignore")

## Imputación Agrupada Directa
Imputamos nulos utilizando la mediana o moda según la escuela y la época. Si algún grupo carece de datos suficientes, utilizamos la mediana/moda global como respaldo inmediato. Mantenemos No information en variables socioeconómicas donde la ausencia de datos es en sí misma una señal.

In [5]:
features_numeric = [
    "age", "PNA", "admission.test", "english.evaluation", "admission.rubric", 
    "general.math.eval", "FTE", "scholarship.perc", "loan.perc", "activity_count_unified"
]

features_categorical = [
    "gender", "foreign", "tec.no.tec", "online.test", "school.cost", 
    "scholarship.type", "max.degree.parents", "parents.exatec", 
    "first.generation", "socioeconomic.level", "social.lag"
]

# Imputación Numérica (Mediana por escuela y era)
for col in features_numeric:
    # Intento de imputación agrupada
    group_median = df.groupby(["school", "era_code"])[col].transform("median")
    df[col] = df[col].fillna(group_median)
    # Respaldo global si el grupo estaba completamente vacío
    df[col] = df[col].fillna(df[col].median())

# Imputación Categórica (Moda por escuela y era)
for col in features_categorical:
    # Reemplazar nulos reales con la categoría 'No information' si aplica
    if col in ["max.degree.parents", "parents.exatec", "first.generation", "socioeconomic.level", "social.lag"]:
        df[col] = df[col].fillna("No information")
    else:
        group_mode = df.groupby(["school", "era_code"])[col].transform(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
        df[col] = df[col].fillna(group_mode)
        df[col] = df[col].fillna(df[col].mode()[0])

## Selección, Exportación y Resumen del Dataset
Selección plana de variables estrictamente necesarias para el modelo. Generamos exportaciones limpias y un bloque de validación visual e informativa para entender la estructura final que consumirá la regresión.

In [6]:
# Lista plana y explícita de features para el modelo
final_features = [
    "dropout", "era_code", "school", "school_code"
] + features_numeric + features_categorical

df_final = df[final_features].copy()

# Exportación simplificada
OUTPUT_NAME = "dataset_estudio_desercion"
df_final.to_csv(OUTPUT_DIR / f"{OUTPUT_NAME}.csv", index=False, encoding="utf-8-sig")
df_final.to_parquet(OUTPUT_DIR / f"{OUTPUT_NAME}.parquet", index=False)

# ── Visualización e Información General ──
print(f"{"="*50}")
print(f"RESUMEN DEL DATASET: {OUTPUT_NAME}")
print(f"{"="*50}")
print(f"Filas totales (Profesional) : {len(df_final):,}")
print(f"Columnas finales            : {len(df_final.columns)}\n")

print("── Porcentaje de Valores Nulos por Columna ──")
missing_pct = (df_final.isna().mean() * 100).round(2)
print(missing_pct[missing_pct > 0].to_string() if missing_pct.sum() > 0 else "0% nulos (Imputación completa y exitosa).")
print("\n")

print("── Tipos de Datos ──")
print(df_final.dtypes.value_counts().to_string())
print("\n")

print("── Distribución por Escuela y Época (Tasa de Deserción) ──")
group_stats = df_final.groupby(["school", "era_code"]).agg(
    n_estudiantes=("dropout", "count"),
    tasa_desercion=("dropout", "mean")
).reset_index()

group_stats["era"] = group_stats["era_code"].map({0: "Pre-Tec21", 1: "Tec21"})
group_stats["tasa_desercion"] = (group_stats["tasa_desercion"] * 100).round(2).astype(str) + "%"

# Formato de tabla para la consola
display(group_stats[["school", "era", "n_estudiantes", "tasa_desercion"]].sort_values(["school", "era"]))

RESUMEN DEL DATASET: dataset_estudio_desercion
Filas totales (Profesional) : 77,517
Columnas finales            : 25

── Porcentaje de Valores Nulos por Columna ──
0% nulos (Imputación completa y exitosa).


── Tipos de Datos ──
str        11
float64     8
int64       4
int8        1
int16       1


── Distribución por Escuela y Época (Tasa de Deserción) ──


,school,era,n_estudiantes,tasa_desercion
0,EAAD-Engineering and Sciences,Tec21,747,6.83%
1,ECSG,Pre-Tec21,3788,9.03%
2,ECSG,Tec21,1596,9.4%
3,EHE-EAAD,Pre-Tec21,7314,10.94%
4,EHE-EAAD,Tec21,3003,10.89%
5,EIC,Pre-Tec21,24381,7.97%
6,EIC,Tec21,10756,7.87%
7,EMCS,Pre-Tec21,2203,10.44%
8,EMCS,Tec21,1159,13.46%
9,EN,Pre-Tec21,15324,8.94%


### Feature Manifest

Hace más facil luego tener seguimiento de que se está guardando en el dataset.

In [7]:
FEATURE_MANIFEST = {
    "target": "dropout",
    "group_vars": {
        "era_code": "int8 - 0: Pre-Tec21 (AD14-AD18), 1: Tec21 (AD19-AD20)",
        "school_code": "int16 - Índice entero basado en las escuelas restantes de Profesional"
    },
    "numeric_common": features_numeric,
    "categorical_common": features_categorical,
    "categorical_domains": {
        "gender": ["Male", "Female"],
        "foreign": ["Local", "Yes: National", "Yes: Foreigner"],
        "tec.no.tec": ["TEC", "NO TEC"],
        "online.test": ["0", "1"],
        "school.cost": ["Public", "Low cost", "Medium cost", "Medium-high cost", "High cost", "Not defined"],
        "scholarship.type": [
            "Academic talent", "Army/Navy scholarship", "Child of Professor/Employee/Director",
            "Contingency scholarship", "Cultural talent", "Entrepreneurial talent", 
            "Leaders of Tomorrow Scholarship", "Leadership talent", "No scholarship", 
            "Sports Talent", "Traditional"
        ],
        "max.degree.parents": ["No information", "No degree", "Undergraduate degree", "Master degree", "PhD"],
        "parents.exatec": ["Yes", "No", "No information"],
        "first.generation": ["Yes", "No", "No information", "Does not apply"],
        "socioeconomic.level": ["Level 1", "Level 2", "Level 3", "Level 4", "Level 5", "Level 6", "Level 7", "No information"],
        "social.lag": ["Low", "Medium", "High", "No information"]
    }
}

# Guardar manifiesto en JSON para la celda de modelado
with open(OUTPUT_DIR / "feature_manifest.json", "w", encoding="utf-8") as f:
    json.dump(FEATURE_MANIFEST, f, indent=2, ensure_ascii=False)

### Missingness

In [9]:
# %% [6] ── Audit & Structural Missingness Analysis ───────────────────────────

print(f"AUDITORÍA FINAL DEL DATASET: {OUTPUT_NAME}")
print(f"Filas totales (Profesional) : {len(df_final):,}")
print(f"Columnas finales            : {len(df_final.columns)}")
print()

print("Atributos Categóricos Registrados:")
for col, values in FEATURE_MANIFEST["categorical_domains"].items():
    print(f"  * {col:<20} -> {', '.join(values)}")
print()

print("Auditoría de Valores Nulos (Missingness):")
missing_counts = df_final.isna().sum()
missing_cols = missing_counts[missing_counts > 0]

if missing_cols.empty:
    print("0 nulos detectados. Dataset completamente limpio y listo para modelar.")
    print()
else:
    # Faltantes Globales
    missing_stats = pd.DataFrame({
        "Nulos Totales": missing_cols,
        "% del Dataset": (missing_cols / len(df_final) * 100).round(2)
    })
    print("Faltantes Globales por Columna:")
    display(missing_stats.sort_values("% del Dataset", ascending=False))
    print()
    
    # Missingness Estratificado por Época (Detecta fallas estructurales)
    print("Distribución de Nulos por Época (Verificación Estructural):")
    missing_by_era = df_final[missing_cols.index.tolist() + ["era_code"]].groupby("era_code").apply(lambda x: x.isna().sum())
    missing_by_era = missing_by_era.drop(columns=["era_code"], errors="ignore")
    missing_by_era.index = missing_by_era.index.map({0: "Pre-Tec21", 1: "Tec21"})
    
    # Calcular porcentajes relativos a cada época
    era_counts = df_final["era_code"].value_counts().rename(index={0: "Pre-Tec21", 1: "Tec21"})
    missing_by_era_pct = missing_by_era.div(era_counts, axis=0).multiply(100).round(1).astype(str) + "%"
    
    # Combinar conteos y porcentajes
    display_df = missing_by_era.astype(str) + " (" + missing_by_era_pct + ")"
    display(display_df.T)
    print()

print("Distribución por Escuela y Época (Tasa de Deserción):")
group_stats = df_final.groupby(["school", "era_code"]).agg(
    n_estudiantes=("dropout", "count"),
    tasa_desercion=("dropout", "mean")
).reset_index()

group_stats["era"] = group_stats["era_code"].map({0: "Pre-Tec21", 1: "Tec21"})
group_stats["tasa_desercion"] = (group_stats["tasa_desercion"] * 100).round(2).astype(str) + "%"

# Mostrar tabla final
display(group_stats[["school", "era", "n_estudiantes", "tasa_desercion"]].sort_values(["school", "era"]))

AUDITORÍA FINAL DEL DATASET: dataset_estudio_desercion
Filas totales (Profesional) : 77,517
Columnas finales            : 25

Atributos Categóricos Registrados:
  * gender               -> Male, Female
  * foreign              -> Local, Yes: National, Yes: Foreigner
  * tec.no.tec           -> TEC, NO TEC
  * online.test          -> 0, 1
  * school.cost          -> Public, Low cost, Medium cost, Medium-high cost, High cost, Not defined
  * scholarship.type     -> Academic talent, Army/Navy scholarship, Child of Professor/Employee/Director, Contingency scholarship, Cultural talent, Entrepreneurial talent, Leaders of Tomorrow Scholarship, Leadership talent, No scholarship, Sports Talent, Traditional
  * max.degree.parents   -> No information, No degree, Undergraduate degree, Master degree, PhD
  * parents.exatec       -> Yes, No, No information
  * first.generation     -> Yes, No, No information, Does not apply
  * socioeconomic.level  -> Level 1, Level 2, Level 3, Level 4, Level 5, Leve

,Nulos Totales,% del Dataset



Distribución de Nulos por Época (Verificación Estructural):


era_code,Pre-Tec21,Tec21



Distribución por Escuela y Época (Tasa de Deserción):


,school,era,n_estudiantes,tasa_desercion
0,EAAD-Engineering and Sciences,Tec21,747,6.83%
1,ECSG,Pre-Tec21,3788,9.03%
2,ECSG,Tec21,1596,9.4%
3,EHE-EAAD,Pre-Tec21,7314,10.94%
4,EHE-EAAD,Tec21,3003,10.89%
5,EIC,Pre-Tec21,24381,7.97%
6,EIC,Tec21,10756,7.87%
7,EMCS,Pre-Tec21,2203,10.44%
8,EMCS,Tec21,1159,13.46%
9,EN,Pre-Tec21,15324,8.94%
